# ReliableVAEP / RAR Extension: Stable-League Candidate

## FA Women's Super League, 2018/19–2020/21

This corrected standalone Kaggle notebook uses the **actual StatsBomb Open Data
coverage** available at runtime.

The previous version stopped because it required exact equality between the
official number of played matches and the number currently present in the
StatsBomb Open Data file. For 2018/19 Kaggle returned **107 StatsBomb matches
versus 110 officially played matches**.

This version uses a predeclared source-coverage rule instead:

- compare each season with its official played-match count;
- require at least **95% source coverage** for every season;
- record and export the exact observed match count and coverage percentage;
- continue when coverage is high but slightly incomplete;
- stop if any season falls below 95%.

The stability and RAR methodology is unchanged.

### Analysis settings

- Base eligibility: ≥5 matches, ≥300 minutes, ≥150 actions
- Longitudinal league cohort: ≥45 appearances
- Raw stability repetitions: 200 per `n`
- Stability target: median Spearman ≥0.80
- Sustained criterion: 3 consecutive `n` values
- Full ReliableVAEP bootstrap: 3,500 draws
- Pairwise support threshold: 0.90
- Bradley-Terry ridge: 1e-4
- RAR validation: `n = 5, 10, 15, 20`
- RAR validation repetitions: 200
- Inner bootstrap: 300 draws per side


> **Canonical ReliableVAEP bootstrap setting for the final SSAC analysis:** `FULL_BOOTSTRAP_B = 3500`.
>
> The value is not an arbitrary round number. ReliableVAEP treats a pairwise ordering as supported at
> \(P_{ij}\ge 0.90\) (or \(\le 0.10\)). Requiring a 95% Wilson Monte Carlo half-width of at most
> 0.01 at \(p=0.90\) gives a minimum of 3,461 resamples. The final analyses therefore use 3,500
> player-preserving match-level bootstrap draws, the smallest 100-draw increment above that bound.
> The completed cross-seed sensitivity test confirmed very high rank-order and pair-classification
> stability at \(B=3500\) across the study datasets.
>
> **Important:** `INNER_BOOTSTRAP_B` values used inside repeated split-sample validation are left at
> their original values. They are nested computational approximations for hundreds of repeated
> validation runs and are not used to produce the paper's primary full-data rank intervals or
> 0.90 pairwise-support decisions.

In [1]:
# FINAL SSAC RUN CONFIGURATION CHECK
CANONICAL_FULL_BOOTSTRAP_B = 3500
print("ReliableVAEP final SSAC run")
print("Canonical full-data bootstrap draws:", CANONICAL_FULL_BOOTSTRAP_B)
print("Run this notebook from top to bottom and use only the newly generated outputs.")


ReliableVAEP final SSAC run
Canonical full-data bootstrap draws: 3500
Run this notebook from top to bottom and use only the newly generated outputs.


In [2]:

# ================================================================
# CELL 1 — KAGGLE-SAFE ISOLATED ENVIRONMENT
# ================================================================
#
# IMPORTANT:
# Kaggle Internet must be ON.
#
# This setup:
#   1. uses /kaggle/working only;
#   2. installs only the small pure-Python "virtualenv" helper into
#      Kaggle's host environment if it is missing;
#   3. creates /kaggle/working/rv_vaep_env;
#   4. installs the complete VAEP stack INSIDE that environment;
#   5. leaves Kaggle's NumPy/pandas/scipy/sklearn untouched.
#
# No runtime restart is required.
# ================================================================

import importlib.util
import os
import shutil
import subprocess
import sys
from pathlib import Path

if not Path("/kaggle/working").exists():
    raise RuntimeError(
        "This notebook is the Kaggle-specific version, but /kaggle/working "
        "was not found. Please run it inside a Kaggle Notebook."
    )

if sys.version_info >= (3, 13):
    raise RuntimeError(
        "Please use a Kaggle Python runtime <= 3.12. "
        "socceraction 1.5.3 is not designed for Python 3.13."
    )

BASE_DIR = Path("/kaggle/working")
ENV_DIR = BASE_DIR / "rv_vaep_env"
ENV_PY = ENV_DIR / "bin" / "python"
ENV_PIP = ENV_DIR / "bin" / "pip"
READY = ENV_DIR / ".reliablevaep_wsl_ready_v1"

IMPORT_TEST = r"""
import numpy
import pandas
import scipy
import sklearn
import xgboost
import socceraction

from socceraction.data.statsbomb import StatsBombLoader
from socceraction.vaep import VAEP
import socceraction.spadl as spadl

print("numpy        =", numpy.__version__)
print("pandas       =", pandas.__version__)
print("scipy        =", scipy.__version__)
print("sklearn      =", sklearn.__version__)
print("xgboost      =", xgboost.__version__)
print("socceraction =", getattr(socceraction, "__version__", "installed"))

# NumPy/pandas binary compatibility smoke test.
df = pandas.DataFrame({"x": numpy.arange(5, dtype=float)})
assert float(df["x"].sum()) == 10.0

print("KAGGLE_ISOLATED_ENV_OK")
"""


def run_capture(args, timeout=None):
    clean_env = os.environ.copy()
    clean_env.pop("PYTHONPATH", None)
    clean_env["PYTHONNOUSERSITE"] = "1"
    return subprocess.run(
        [str(x) for x in args],
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        timeout=timeout,
        env=clean_env,
    )


def environment_is_healthy():
    if not (
        ENV_PY.exists()
        and ENV_PIP.exists()
        and READY.exists()
    ):
        return False

    test = run_capture(
        [ENV_PY, "-c", IMPORT_TEST]
    )

    if (
        test.returncode == 0
        and "KAGGLE_ISOLATED_ENV_OK" in test.stdout
    ):
        print("Existing isolated environment verified:")
        print(test.stdout)
        return True

    print("Existing isolated environment is incomplete/unhealthy.")
    print(test.stdout)
    return False


if not environment_is_healthy():

    # Remove only our own previous environment.
    if ENV_DIR.exists():
        print("Removing incomplete ReliableVAEP environment:", ENV_DIR)
        shutil.rmtree(ENV_DIR, ignore_errors=True)

    # ------------------------------------------------------------
    # Install virtualenv only if unavailable.
    # virtualenv is pure Python and does not alter NumPy/pandas.
    # ------------------------------------------------------------
    if importlib.util.find_spec("virtualenv") is None:
        print("Installing the lightweight virtualenv helper into Kaggle...")
        helper = run_capture([
            sys.executable,
            "-m",
            "pip",
            "install",
            "--quiet",
            "--no-cache-dir",
            "virtualenv==20.28.0",
        ])

        print(helper.stdout)

        if helper.returncode != 0:
            raise RuntimeError(
                "Could not install virtualenv.\n"
                "Make sure Kaggle Internet is ON.\n\n"
                + helper.stdout
            )
    else:
        print("virtualenv helper is already available.")

    # ------------------------------------------------------------
    # Create isolated environment using virtualenv.
    # This avoids system ensurepip entirely.
    # ------------------------------------------------------------
    print("\nCreating isolated environment:", ENV_DIR)

    create = run_capture([
        sys.executable,
        "-m",
        "virtualenv",
        "--clear",
        str(ENV_DIR),
    ])

    print(create.stdout)

    if (
        create.returncode != 0
        or not ENV_PY.exists()
        or not ENV_PIP.exists()
    ):
        raise RuntimeError(
            "virtualenv could not create the isolated environment.\n\n"
            + create.stdout
        )

    # ------------------------------------------------------------
    # Install one internally consistent stack.
    # These packages exist ONLY inside /kaggle/working/rv_vaep_env.
    # ------------------------------------------------------------
    packages = [
        "numpy==1.26.4",
        "pandas==2.1.4",
        "scipy==1.13.1",
        "scikit-learn==1.5.2",
        "xgboost==2.1.4",
        "matplotlib==3.9.2",
        "tqdm==4.66.5",
        "multimethod==1.10",
        "pandera==0.17.2",
        "socceraction[statsbomb]==1.5.3",
    ]

    print("\nInstalling the isolated ReliableVAEP stack...")
    print("This can take a few minutes on the first run.")

    install = run_capture([
        ENV_PY,
        "-m",
        "pip",
        "install",
        "--no-cache-dir",
        *packages,
    ])

    print(install.stdout)

    if install.returncode != 0:
        raise RuntimeError(
            "Package installation failed INSIDE the isolated environment.\n"
            "Kaggle's host NumPy/pandas environment has not been changed.\n\n"
            + install.stdout
        )

    # ------------------------------------------------------------
    # Verify dependency consistency.
    # ------------------------------------------------------------
    print("\nChecking isolated dependency consistency...")

    check = run_capture([
        ENV_PY,
        "-m",
        "pip",
        "check",
    ])

    print(
        check.stdout
        if check.stdout.strip()
        else "pip check: no dependency conflicts"
    )

    if check.returncode != 0:
        raise RuntimeError(
            "Dependency conflicts remain inside the isolated environment.\n\n"
            + check.stdout
        )

    # ------------------------------------------------------------
    # Import and binary compatibility test.
    # ------------------------------------------------------------
    print("\nVerifying isolated imports...")

    verify = run_capture([
        ENV_PY,
        "-c",
        IMPORT_TEST,
    ])

    print(verify.stdout)

    if (
        verify.returncode != 0
        or "KAGGLE_ISOLATED_ENV_OK" not in verify.stdout
    ):
        raise RuntimeError(
            "The isolated environment was created, but its scientific stack "
            "failed verification.\n\n"
            + verify.stdout
        )

    READY.write_text("ready", encoding="utf-8")

    print("\nKaggle isolated environment setup completed successfully.")

else:
    print("\nReusing the verified Kaggle isolated environment.")

print("\nAnalysis Python:", ENV_PY)
print("Kaggle host scientific packages were NOT downgraded or replaced.")


Installing the lightweight virtualenv helper into Kaggle...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.3/4.3 MB 80.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 470.6/470.6 kB 239.5 MB/s eta 0:00:00


Creating isolated environment: /kaggle/working/rv_vaep_env
created virtual environment CPython3.12.13.final.0-64 in 445ms
  creator CPython3Posix(dest=/kaggle/working/rv_vaep_env, clear=True, no_vcs_ignore=False, global=False)
  seeder FromAppData(download=False, pip=bundle, via=copy, app_data_dir=/root/.local/share/virtualenv)
    added seed packages: pip==24.3.1
  activators BashActivator,CShellActivator,FishActivator,NushellActivator,PowerShellActivator,PythonActivator


Installing the isolated ReliableVAEP stack...
This can take a few minutes on the first run.
Error in sitecustomize; set PYTHONVERBOSE for traceback:
ModuleNotFoundError: No module named 'wrapt'
INFO: pip is looking at multiple versions of contourpy to determine which version is compatible with other 

## Analysis

The next cell writes the complete WSL longitudinal analysis script. No files from the previous ReliableVAEP notebooks are required.


In [3]:
# ================================================================
# CELL 3 — WRITE THE COMPLETE STANDALONE WSL ANALYSIS SCRIPT
# ================================================================
from pathlib import Path

SCRIPT_PATH = Path("/kaggle/working/reliablevaep_wsl_longitudinal_analysis.py")

ANALYSIS_SCRIPT = '\n# ================================================================\n# ReliableVAEP — WSL 2018/19-2020/21 Longitudinal League PCS/RAR Test\n# FULLY STANDALONE KAGGLE ANALYSIS\n# ================================================================\n\nimport json\nimport os\nimport shutil\nimport time\nimport urllib.request\nimport warnings\nfrom pathlib import Path\n\n# Kaggle exports a notebook-only matplotlib backend.  The analysis\n# runs in an isolated subprocess, so force a normal file backend first.\nos.environ["MPLBACKEND"] = "Agg"\n\nimport matplotlib\nmatplotlib.use("Agg", force=True)\n\nimport matplotlib.pyplot as plt\nimport numpy as np\nimport pandas as pd\nimport xgboost as xgb\n\nfrom scipy.optimize import minimize\nfrom scipy.special import expit, logit\nfrom scipy.stats import spearmanr, wilcoxon\nfrom sklearn.linear_model import LogisticRegression\nfrom sklearn.metrics import average_precision_score, brier_score_loss, roc_auc_score\nfrom sklearn.model_selection import GroupKFold, GroupShuffleSplit\nfrom tqdm.auto import tqdm\n\nfrom socceraction.data.statsbomb import StatsBombLoader\nimport socceraction.spadl as spadl\nfrom socceraction.vaep import VAEP\nfrom socceraction.vaep import formula as vaep_formula\n\nwarnings.filterwarnings("ignore")\n\n# ----------------------------------------------------------------\n# Reproducibility and analysis settings\n# ----------------------------------------------------------------\nSEED = 20260905\nnp.random.seed(SEED)\n\nCOMPETITION_ID = 37\n\nSEASON_SPECS = [\n    # expected_matches = official played-match count for context.\n    # StatsBomb Open Data may omit a small number of matches, so source\n    # completeness is checked with MIN_SOURCE_COVERAGE rather than exact equality.\n    {"season_id": 4,  "season_name": "2018/2019", "expected_matches": 110},\n    {"season_id": 42, "season_name": "2019/2020", "expected_matches": 87},\n    {"season_id": 90, "season_name": "2020/2021", "expected_matches": 131},\n]\n\nMIN_SOURCE_COVERAGE = 0.95\n\nDATASET_LABEL = "FA Women\'s Super League 2018/19-2020/21 longitudinal league cohort"\n\n# Base eligibility keeps the canonical ReliableVAEP thresholds.\nMIN_MATCHES = 5\nMIN_MINUTES = 300\nMIN_ACTIONS = 150\n\n# The stable-league hypothesis is tested on a predeclared high-exposure\n# league-level cohort. 45 appearances permit two non-overlapping samples\n# up to n=22 while retaining a multi-club ranking universe.\nLONGITUDINAL_MIN_MATCHES = 45\nSTABILITY_MIN_PLAYERS = 12\nSTABILITY_MIN_N = 5\nSTABILITY_TARGET = 0.80\nSUSTAINED_POINTS = 3\nSTABILITY_REPS = 200\n\n# Full static ReliableVAEP / PCS / RAR analysis on the same longitudinal cohort.\nFULL_BOOTSTRAP_B = 3500\nPAIRWISE_SUPPORT = 0.90\nBT_RIDGE = 1e-4\n\n# Independent RAR validation uses the same >=45-match cohort.\nVALIDATION_MIN_MATCHES = LONGITUDINAL_MIN_MATCHES\nVALIDATION_N_VALUES = [5, 10, 15, 20]\nVALIDATION_REPS = 200\nINNER_BOOTSTRAP_B = 300\n\nOUTER_FOLDS = 5\nCALIBRATION_MATCH_FRACTION = 0.20\n\nROOT = Path("/kaggle/working")\nCACHE_ROOT = ROOT / "rv_wsl_1819_2021_cache"\nOUT_ROOT = ROOT / "rv_wsl_1819_2021_pcs_rar_outputs"\n\nCACHE_ROOT.mkdir(parents=True, exist_ok=True)\nOUT_ROOT.mkdir(parents=True, exist_ok=True)\n\nBASE_URL = "https://raw.githubusercontent.com/hudl/open-data/master/data"\n\nprint("=" * 96)\nprint("ReliableVAEP WSL 2018/19-2020/21 longitudinal league stability + PCS/RAR")\nprint("=" * 96)\nprint("Competition ID:", COMPETITION_ID)\nprint("Seasons:", ", ".join(s["season_name"] for s in SEASON_SPECS))\nprint("Official played matches for coverage reference:", sum(s["expected_matches"] for s in SEASON_SPECS))\nprint("Minimum accepted StatsBomb source coverage:", f"{100*MIN_SOURCE_COVERAGE:.1f}%")\nprint("Base eligibility:",\n      f">={MIN_MATCHES} matches, >={MIN_MINUTES} minutes, >={MIN_ACTIONS} actions")\nprint("Longitudinal ranking cohort: >=", LONGITUDINAL_MIN_MATCHES, "appearances")\nprint("Raw-stability target:",\n      f"median Spearman >= {STABILITY_TARGET:.2f} for {SUSTAINED_POINTS} consecutive n values")\nprint("Full bootstrap draws:", FULL_BOOTSTRAP_B)\nprint("Pairwise support threshold:", PAIRWISE_SUPPORT)\nprint("RAR validation n values:", VALIDATION_N_VALUES)\nprint("RAR validation repetitions per n:", VALIDATION_REPS)\nprint()\n\n# ================================================================\n# Utility functions\n# ================================================================\n\ndef get_json(url, path, retries=5):\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n\n    if path.exists():\n        try:\n            with open(path, encoding="utf-8") as f:\n                return json.load(f)\n        except Exception:\n            path.unlink(missing_ok=True)\n\n    for attempt in range(1, retries + 1):\n        try:\n            req = urllib.request.Request(\n                url,\n                headers={"User-Agent": "ReliableVAEP-WSL-Longitudinal/1.0"},\n            )\n            with urllib.request.urlopen(req, timeout=120) as response:\n                obj = json.loads(response.read().decode("utf-8"))\n            with open(path, "w", encoding="utf-8") as f:\n                json.dump(obj, f)\n            return obj\n        except Exception as exc:\n            if attempt == retries:\n                raise RuntimeError(f"Download failed: {url}") from exc\n            time.sleep(min(2 ** attempt, 15))\n\n\ndef make_model(seed):\n    return xgb.XGBClassifier(\n        n_estimators=180,\n        max_depth=3,\n        learning_rate=0.05,\n        min_child_weight=5,\n        subsample=0.85,\n        colsample_bytree=0.85,\n        reg_lambda=2.0,\n        objective="binary:logistic",\n        eval_metric="logloss",\n        tree_method="hist",\n        random_state=seed,\n        n_jobs=-1,\n    )\n\n\ndef fit_platt(prob, y):\n    prob = np.asarray(prob, dtype=float)\n    y = np.asarray(y, dtype=int)\n    if len(np.unique(y)) < 2:\n        return None\n    z = logit(np.clip(prob, 1e-6, 1 - 1e-6)).reshape(-1, 1)\n    model = LogisticRegression(solver="lbfgs", max_iter=1000)\n    model.fit(z, y)\n    return model\n\n\ndef apply_platt(model, prob):\n    prob = np.asarray(prob, dtype=float)\n    if model is None:\n        return prob.copy()\n    z = logit(np.clip(prob, 1e-6, 1 - 1e-6)).reshape(-1, 1)\n    return model.predict_proba(z)[:, 1]\n\n\ndef safe_auc(y, p):\n    return float(roc_auc_score(y, p)) if len(np.unique(y)) > 1 else np.nan\n\n\ndef baseline_brier(y):\n    y = np.asarray(y, dtype=float)\n    return float(np.mean((y - y.mean()) ** 2))\n\n\ndef vaep90(q):\n    minutes = float(q["minutes_played"].sum())\n    if minutes <= 0:\n        return np.nan\n    return float(90.0 * q["vaep"].sum() / minutes)\n\n\ndef ranks_desc(values):\n    return (\n        pd.Series(np.asarray(values, dtype=float))\n        .rank(ascending=False, method="average")\n        .to_numpy(dtype=float)\n    )\n\n\ndef player_id_to_jsonable(x):\n    if isinstance(x, (np.integer, int)):\n        return int(x)\n    try:\n        xf = float(x)\n        if xf.is_integer():\n            return int(xf)\n        return xf\n    except Exception:\n        return str(x)\n\n\nSBL = StatsBombLoader(getter="remote", creds=None)\n\n\n# ================================================================\n# STEP 1 — Canonical WSL 2018/19-2020/21 player-match VAEP\n# ================================================================\n\ndef build_wsl_player_match():\n    dataset_dir = OUT_ROOT / "canonical_build"\n    dataset_dir.mkdir(parents=True, exist_ok=True)\n\n    marker = dataset_dir / "BUILD_COMPLETE.json"\n    pm_pickle = dataset_dir / "player_match_vaep.pkl"\n    cohort_csv = dataset_dir / "eligible_cohort.csv"\n    metrics_csv = dataset_dir / "crossfit_model_metrics.csv"\n\n    if marker.exists() and pm_pickle.exists() and cohort_csv.exists() and metrics_csv.exists():\n        try:\n            meta = json.loads(marker.read_text(encoding="utf-8"))\n            pm = pd.read_pickle(pm_pickle)\n            cohort = pd.read_csv(cohort_csv)\n            metrics = pd.read_csv(metrics_csv)\n            print("Loading completed cached canonical WSL longitudinal build.")\n            print("Matches:", meta["matches"])\n            print("Base eligible players:", meta["base_eligible_players"])\n            return pm, cohort, metrics, meta\n        except Exception as exc:\n            print("Cached build could not be loaded; rebuilding:", exc)\n\n    # ------------------------------------------------------------\n    # Match lists + explicit coverage audit\n    # ------------------------------------------------------------\n    game_parts = []\n    coverage_rows = []\n\n    for spec in SEASON_SPECS:\n        sid = int(spec["season_id"])\n        sname = str(spec["season_name"])\n        expected = int(spec["expected_matches"])\n\n        match_file = CACHE_ROOT / "matches" / f"{COMPETITION_ID}_{sid}.json"\n        matches_raw = get_json(\n            f"{BASE_URL}/matches/{COMPETITION_ID}/{sid}.json",\n            match_file,\n        )\n\n        g = pd.DataFrame([\n            {\n                "game_id": int(m["match_id"]),\n                "game_date": pd.to_datetime(m["match_date"]),\n                "season_id": sid,\n                "season_name": sname,\n                "home_team_id": int(m["home_team"]["home_team_id"]),\n                "away_team_id": int(m["away_team"]["away_team_id"]),\n                "home_team_name": str(m["home_team"]["home_team_name"]),\n                "away_team_name": str(m["away_team"]["away_team_name"]),\n            }\n            for m in matches_raw\n        ]).sort_values(["game_date", "game_id"]).reset_index(drop=True)\n\n        observed = int(g["game_id"].nunique())\n\n        coverage_fraction = observed / expected if expected > 0 else np.nan\n        coverage_ok = bool(coverage_fraction >= MIN_SOURCE_COVERAGE)\n\n        coverage_rows.append({\n            "season_id": sid,\n            "season_name": sname,\n            "official_played_matches": expected,\n            "observed_statsbomb_matches": observed,\n            "coverage_fraction": coverage_fraction,\n            "coverage_ok": coverage_ok,\n        })\n\n        print(\n            f"WSL {sname}: StatsBomb matches found = {observed} "\n            f"(official played = {expected}; coverage = {100*coverage_fraction:.1f}%)"\n        )\n\n        if not coverage_ok:\n            raise RuntimeError(\n                f"Coverage audit failed for WSL {sname}: StatsBomb contains "\n                f"{observed}/{expected} matches ({100*coverage_fraction:.1f}%), "\n                f"below the predeclared minimum source coverage of "\n                f"{100*MIN_SOURCE_COVERAGE:.1f}%."\n            )\n\n        if observed != expected:\n            print(\n                f"WARNING: WSL {sname} is not complete in the current StatsBomb "\n                f"Open Data file ({observed}/{expected}). Analysis will use the "\n                "available source matches and will report the exact coverage."\n            )\n\n        game_parts.append(g)\n\n    games = (\n        pd.concat(game_parts, ignore_index=True)\n        .sort_values(["game_date", "game_id"])\n        .reset_index(drop=True)\n    )\n\n    if games["game_id"].duplicated().any():\n        dup = games.loc[games["game_id"].duplicated(), "game_id"].tolist()\n        raise RuntimeError(f"Duplicate match IDs across seasons: {dup[:10]}")\n\n    expected_total = int(sum(s["expected_matches"] for s in SEASON_SPECS))\n    observed_total = int(games["game_id"].nunique())\n    overall_coverage = observed_total / expected_total if expected_total > 0 else np.nan\n\n    if overall_coverage < MIN_SOURCE_COVERAGE:\n        raise RuntimeError(\n            f"Longitudinal source coverage is only "\n            f"{observed_total}/{expected_total} "\n            f"({100*overall_coverage:.1f}%), below the required "\n            f"{100*MIN_SOURCE_COVERAGE:.1f}%."\n        )\n\n    print(\n        "Coverage audit passed. "\n        f"StatsBomb sample = {observed_total}/{expected_total} official played "\n        f"matches ({100*overall_coverage:.1f}% overall coverage)."\n    )\n\n    pd.DataFrame(coverage_rows).to_csv(\n        dataset_dir / "coverage_audit.csv",\n        index=False,\n    )\n    games.to_csv(dataset_dir / "game_list.csv", index=False)\n\n    # ------------------------------------------------------------\n    # SPADL actions + official player minutes\n    # ------------------------------------------------------------\n    store = {}\n\n    for g in tqdm(\n        games.itertuples(index=False),\n        total=len(games),\n        desc="WSL longitudinal: SPADL",\n        unit="match",\n    ):\n        gid = int(g.game_id)\n\n        action_path = CACHE_ROOT / "spadl" / f"{gid}.pkl"\n        player_path = CACHE_ROOT / "players" / f"{gid}.pkl"\n        action_path.parent.mkdir(parents=True, exist_ok=True)\n        player_path.parent.mkdir(parents=True, exist_ok=True)\n\n        if action_path.exists():\n            actions = pd.read_pickle(action_path)\n        else:\n            events = SBL.events(game_id=gid)\n            actions = (\n                spadl.statsbomb.convert_to_actions(\n                    events=events,\n                    home_team_id=int(g.home_team_id),\n                )\n                .reset_index(drop=True)\n            )\n            actions.to_pickle(action_path)\n\n        if player_path.exists():\n            players = pd.read_pickle(player_path)\n        else:\n            players = SBL.players(game_id=gid)\n            players.to_pickle(player_path)\n\n        if len(actions) == 0:\n            raise RuntimeError(f"No SPADL actions in match {gid}.")\n\n        store[gid] = {\n            "game": pd.Series({\n                "game_id": gid,\n                "home_team_id": int(g.home_team_id),\n                "away_team_id": int(g.away_team_id),\n                "game_date": g.game_date,\n                "season_id": int(g.season_id),\n                "season_name": str(g.season_name),\n            }),\n            "actions": actions,\n            "players": players,\n        }\n\n    # ------------------------------------------------------------\n    # Canonical VAEP features and labels\n    # ------------------------------------------------------------\n    template = VAEP()\n\n    X_parts = []\n    Y_parts = []\n    row_parts = []\n\n    for gid, item in tqdm(store.items(), desc="Canonical VAEP features", unit="match"):\n        Xg = template.compute_features(\n            item["game"],\n            item["actions"],\n        ).reset_index(drop=True)\n\n        Yg = template.compute_labels(\n            item["game"],\n            item["actions"],\n        ).reset_index(drop=True)\n\n        if len(Xg) != len(item["actions"]) or len(Yg) != len(item["actions"]):\n            raise RuntimeError(f"Feature/label/action alignment failed in match {gid}.")\n\n        X_parts.append(Xg)\n        Y_parts.append(Yg)\n        row_parts.append(\n            pd.DataFrame({\n                "game_id": gid,\n                "action_row": np.arange(len(Xg)),\n            })\n        )\n\n    X = pd.concat(X_parts, ignore_index=True)\n    Y = pd.concat(Y_parts, ignore_index=True)\n    ROW = pd.concat(row_parts, ignore_index=True)\n\n    # Convert any nonnumeric encoded feature columns to numeric forms.\n    for col in X.columns:\n        if pd.api.types.is_bool_dtype(X[col]):\n            X[col] = X[col].astype(np.uint8)\n        elif isinstance(X[col].dtype, pd.CategoricalDtype):\n            X[col] = X[col].cat.codes.astype(np.int32)\n        elif pd.api.types.is_object_dtype(X[col]):\n            X[col] = pd.factorize(X[col].astype(str))[0].astype(np.int32)\n\n    bad_features = [c for c in X.columns if not pd.api.types.is_numeric_dtype(X[c])]\n    if bad_features:\n        raise TypeError(f"Non-numeric VAEP feature columns remain: {bad_features}")\n\n    print(\n        f"Action rows={len(X):,}, features={X.shape[1]}, "\n        f"score rate={Y[\'scores\'].mean():.4f}, concede rate={Y[\'concedes\'].mean():.4f}"\n    )\n\n    # ------------------------------------------------------------\n    # Match-level out-of-fold prediction + inner calibration\n    # ------------------------------------------------------------\n    groups = ROW["game_id"].to_numpy()\n    unique_matches = np.unique(groups)\n    n_folds = min(OUTER_FOLDS, len(unique_matches))\n\n    outer = GroupKFold(n_splits=n_folds)\n\n    oof = {\n        "scores_raw": np.full(len(X), np.nan),\n        "scores_cal": np.full(len(X), np.nan),\n        "concedes_raw": np.full(len(X), np.nan),\n        "concedes_cal": np.full(len(X), np.nan),\n    }\n\n    metric_rows = []\n\n    for fold, (train_idx, test_idx) in enumerate(\n        outer.split(X, groups=groups),\n        start=1,\n    ):\n        inner = GroupShuffleSplit(\n            n_splits=1,\n            test_size=CALIBRATION_MATCH_FRACTION,\n            random_state=SEED + fold,\n        )\n\n        fit_rel, cal_rel = next(\n            inner.split(\n                X.iloc[train_idx],\n                groups=groups[train_idx],\n            )\n        )\n\n        fit_idx = train_idx[fit_rel]\n        cal_idx = train_idx[cal_rel]\n\n        row = {\n            "fold": fold,\n            "fit_matches": len(np.unique(groups[fit_idx])),\n            "calibration_matches": len(np.unique(groups[cal_idx])),\n            "test_matches": len(np.unique(groups[test_idx])),\n        }\n\n        for target in ["scores", "concedes"]:\n            model = make_model(\n                SEED + fold * 10 + (1 if target == "scores" else 2)\n            )\n\n            model.fit(\n                X.iloc[fit_idx],\n                Y.iloc[fit_idx][target].astype(int),\n            )\n\n            p_cal_raw = model.predict_proba(X.iloc[cal_idx])[:, 1]\n            calibrator = fit_platt(\n                p_cal_raw,\n                Y.iloc[cal_idx][target].astype(int),\n            )\n\n            p_test_raw = model.predict_proba(X.iloc[test_idx])[:, 1]\n            p_test_cal = apply_platt(calibrator, p_test_raw)\n\n            oof[f"{target}_raw"][test_idx] = p_test_raw\n            oof[f"{target}_cal"][test_idx] = p_test_cal\n\n            y_test = Y.iloc[test_idx][target].astype(int).to_numpy()\n            base = baseline_brier(y_test)\n\n            row[f"{target}_auc"] = safe_auc(y_test, p_test_cal)\n            row[f"{target}_ap"] = float(average_precision_score(y_test, p_test_cal))\n            row[f"{target}_brier_baseline"] = base\n            row[f"{target}_brier_raw"] = float(brier_score_loss(y_test, p_test_raw))\n            row[f"{target}_brier_cal"] = float(brier_score_loss(y_test, p_test_cal))\n            row[f"{target}_brier_skill_cal"] = (\n                1.0 - row[f"{target}_brier_cal"] / base if base > 0 else np.nan\n            )\n\n        metric_rows.append(row)\n\n    for key, values in oof.items():\n        if np.isnan(values).any():\n            raise RuntimeError(f"Incomplete OOF predictions: {key}")\n\n    metrics = pd.DataFrame(metric_rows)\n    metrics.to_csv(metrics_csv, index=False)\n\n    print("\\nCross-fit model means:")\n    print(\n        metrics.mean(numeric_only=True)\n        .round(4)\n        .to_string()\n    )\n\n    # ------------------------------------------------------------\n    # Canonical action VAEP\n    # ------------------------------------------------------------\n    scored_parts = []\n    cursor = 0\n\n    for gid, item in tqdm(store.items(), desc="Canonical action VAEP", unit="match"):\n        actions = item["actions"].reset_index(drop=True)\n        n_actions = len(actions)\n\n        ps = pd.Series(oof["scores_cal"][cursor:cursor + n_actions])\n        pc = pd.Series(oof["concedes_cal"][cursor:cursor + n_actions])\n\n        named = spadl.add_names(actions.copy()).reset_index(drop=True)\n        values = vaep_formula.value(named, ps, pc).reset_index(drop=True)\n\n        z = named.copy()\n        z["game_id"] = gid\n        z["game_date"] = item["game"]["game_date"]\n        z["season_id"] = item["game"]["season_id"]\n        z["season_name"] = item["game"]["season_name"]\n        z["p_score"] = ps.to_numpy()\n        z["p_concede"] = pc.to_numpy()\n\n        for col in ["offensive_value", "defensive_value", "vaep_value"]:\n            z[col] = values[col].to_numpy()\n\n        scored_parts.append(z)\n        cursor += n_actions\n\n    if cursor != len(X):\n        raise RuntimeError("OOF probability/action alignment failed.")\n\n    scored = pd.concat(scored_parts, ignore_index=True)\n\n    # ------------------------------------------------------------\n    # Goal sanity check\n    # ------------------------------------------------------------\n    goal_mask = (\n        scored["type_name"].isin(["shot", "shot_freekick", "shot_penalty"])\n        & scored["result_name"].eq("success")\n    )\n\n    if int(goal_mask.sum()) == 0:\n        raise RuntimeError("No successful goals detected for VAEP sanity checking.")\n\n    goal_values = scored.loc[goal_mask, "vaep_value"]\n    median_goal_vaep = float(goal_values.median())\n    positive_goal_fraction = float((goal_values > 0).mean())\n\n    print(\n        "\\nGoal-value sanity check:",\n        f"n={int(goal_mask.sum())}, "\n        f"median successful-goal VAEP={median_goal_vaep:.4f}, "\n        f"positive fraction={positive_goal_fraction:.3f}"\n    )\n\n    if median_goal_vaep <= 0:\n        raise RuntimeError(\n            "Canonical VAEP sanity check failed: median successful-goal value is not positive."\n        )\n\n    # ------------------------------------------------------------\n    # Player-match aggregation for ALL teams\n    # ------------------------------------------------------------\n    pm_parts = []\n\n    keeper_types = {\n        "keeper_save",\n        "keeper_claim",\n        "keeper_punch",\n        "keeper_pick_up",\n    }\n\n    keeper_action_ids = set(\n        scored.loc[\n            scored["type_name"].isin(keeper_types),\n            "player_id",\n        ]\n        .dropna()\n        .tolist()\n    )\n\n    for gid, item in tqdm(store.items(), desc="Player-match VAEP", unit="match"):\n        players = item["players"].copy()\n        players = players[\n            players["minutes_played"].fillna(0) > 0\n        ].copy()\n\n        ga = scored[scored["game_id"] == gid].copy()\n\n        agg = (\n            ga.dropna(subset=["player_id"])\n            .groupby("player_id", as_index=False)\n            .agg(\n                vaep=("vaep_value", "sum"),\n                offensive_value=("offensive_value", "sum"),\n                defensive_value=("defensive_value", "sum"),\n                actions=("action_id", "size"),\n            )\n        )\n\n        goals = (\n            ga[\n                ga["type_name"].isin(["shot", "shot_freekick", "shot_penalty"])\n                & ga["result_name"].eq("success")\n            ]\n            .dropna(subset=["player_id"])\n            .groupby("player_id")\n            .size()\n            .rename("goals")\n            .reset_index()\n        )\n\n        agg = agg.merge(goals, on="player_id", how="left")\n        agg["goals"] = agg["goals"].fillna(0).astype(int)\n\n        q = players.merge(agg, on="player_id", how="left")\n\n        for col in ["vaep", "offensive_value", "defensive_value"]:\n            q[col] = q[col].fillna(0.0)\n\n        for col in ["actions", "goals"]:\n            q[col] = q[col].fillna(0).astype(int)\n\n        q["game_id"] = gid\n        q["game_date"] = item["game"]["game_date"]\n        q["season_id"] = item["game"]["season_id"]\n        q["season_name"] = item["game"]["season_name"]\n        q["vaep90_match"] = (\n            90.0 * q["vaep"] / q["minutes_played"]\n        )\n\n        if "starting_position_name" in q.columns:\n            q["is_goalkeeper_position"] = (\n                q["starting_position_name"]\n                .astype(str)\n                .str.contains("Goalkeeper", case=False, na=False)\n            )\n        else:\n            q["is_goalkeeper_position"] = False\n\n        q["is_goalkeeper"] = (\n            q["is_goalkeeper_position"].astype(bool)\n            | q["player_id"].isin(keeper_action_ids)\n        )\n\n        pm_parts.append(q)\n\n    pm = pd.concat(pm_parts, ignore_index=True)\n\n    # ------------------------------------------------------------\n    # Eligibility\n    # ------------------------------------------------------------\n    cohort = (\n        pm.groupby(["player_id", "player_name"], as_index=False)\n        .agg(\n            total_matches=("game_id", "nunique"),\n            total_minutes=("minutes_played", "sum"),\n            total_actions=("actions", "sum"),\n            total_vaep=("vaep", "sum"),\n            offensive_value=("offensive_value", "sum"),\n            defensive_value=("defensive_value", "sum"),\n            goals=("goals", "sum"),\n            goalkeeper_match_rows=("is_goalkeeper", "sum"),\n        )\n    )\n\n    cohort["vaep90"] = (\n        90.0 * cohort["total_vaep"] / cohort["total_minutes"]\n    )\n\n    cohort["is_goalkeeper"] = cohort["goalkeeper_match_rows"] > 0\n\n    cohort = cohort[\n        (cohort["total_matches"] >= MIN_MATCHES)\n        & (cohort["total_minutes"] >= MIN_MINUTES)\n        & (cohort["total_actions"] >= MIN_ACTIONS)\n    ].copy()\n\n    cohort = (\n        cohort.sort_values("vaep90", ascending=False)\n        .reset_index(drop=True)\n    )\n\n    cohort["raw_VAEP_rank"] = np.arange(1, len(cohort) + 1)\n\n    print("\\nEligible players:", len(cohort))\n    print("Goalkeepers among eligible players:", int(cohort["is_goalkeeper"].sum()))\n\n    if len(cohort) < 100:\n        print(\n            "WARNING: fewer than 100 base-eligible players were produced. "\n            "The notebook will continue, but inspect the coverage audit and eligibility outputs."\n        )\n\n    pm.to_pickle(pm_pickle)\n    pm.to_csv(dataset_dir / "player_match_vaep.csv", index=False)\n    cohort.to_csv(cohort_csv, index=False)\n\n    meta = {\n        "competition_id": COMPETITION_ID,\n        "season_ids": [int(s["season_id"]) for s in SEASON_SPECS],\n        "season_names": [str(s["season_name"]) for s in SEASON_SPECS],\n        "official_played_matches": int(expected_total),\n        "statsbomb_matches": int(observed_total),\n        "source_coverage_fraction": float(overall_coverage),\n        "base_eligible_players": int(len(cohort)),\n        "base_eligible_goalkeepers": int(cohort["is_goalkeeper"].sum()),\n        "median_successful_goal_vaep": median_goal_vaep,\n        "positive_goal_fraction": positive_goal_fraction,\n    }\n\n    marker.write_text(json.dumps(meta, indent=2), encoding="utf-8")\n\n    return pm, cohort, metrics, meta\n\n\n\n# ================================================================\n# PRE-RAR LONGITUDINAL LEAGUE STABILITY\n# ================================================================\n\ndef longitudinal_raw_stability(pm, base_cohort, repeats=STABILITY_REPS, seed=SEED + 7000):\n    fixed = base_cohort[\n        base_cohort["total_matches"] >= LONGITUDINAL_MIN_MATCHES\n    ].copy()\n\n    fixed = fixed.sort_values(\n        ["vaep90", "player_name"],\n        ascending=[False, True],\n    ).reset_index(drop=True)\n    fixed["raw_VAEP_rank"] = np.arange(1, len(fixed) + 1)\n\n    if len(fixed) < STABILITY_MIN_PLAYERS:\n        raise RuntimeError(\n            f"Only {len(fixed)} players have at least {LONGITUDINAL_MIN_MATCHES} "\n            f"appearances across the three WSL seasons. Need at least "\n            f"{STABILITY_MIN_PLAYERS} for the declared league-level stability cohort."\n        )\n\n    player_ids = fixed["player_id"].tolist()\n\n    grouped = {\n        pid: (\n            q.sort_values(["game_date", "game_id"])\n            .reset_index(drop=True)\n        )\n        for pid, q in pm[\n            pm["player_id"].isin(player_ids)\n        ].groupby("player_id")\n    }\n\n    min_observed = min(len(grouped[pid]) for pid in player_ids)\n    n_max = min_observed // 2\n\n    if n_max < STABILITY_MIN_N:\n        raise RuntimeError(\n            f"The fixed cohort supports only n_max={n_max}; "\n            f"STABILITY_MIN_N={STABILITY_MIN_N}."\n        )\n\n    n_values = list(range(STABILITY_MIN_N, n_max + 1))\n    rng = np.random.default_rng(seed)\n    rows = []\n\n    print("\\n" + "=" * 96)\n    print("PRE-RAR LONGITUDINAL RAW-VAEP STABILITY")\n    print("=" * 96)\n    print("Fixed league-level cohort players:", len(player_ids))\n    print("Minimum observed appearances in cohort:", min_observed)\n    print("Tested n range:", n_values[0], "to", n_values[-1])\n    print("Repetitions per n:", repeats)\n\n    for n in n_values:\n        for rep in range(repeats):\n            left_values = np.empty(len(player_ids), dtype=float)\n            right_values = np.empty(len(player_ids), dtype=float)\n\n            for j, pid in enumerate(player_ids):\n                q = grouped[pid]\n                chosen = rng.choice(len(q), size=2 * n, replace=False)\n                qA = q.iloc[chosen[:n]]\n                qB = q.iloc[chosen[n:]]\n                left_values[j] = vaep90(qA)\n                right_values[j] = vaep90(qB)\n\n            left_rank = ranks_desc(left_values)\n            right_rank = ranks_desc(right_values)\n\n            rho = float(spearmanr(left_rank, right_rank).statistic)\n\n            rows.append({\n                "n_matches_per_sample": int(n),\n                "rep": int(rep),\n                "players": len(player_ids),\n                "spearman": rho,\n            })\n\n    raw = pd.DataFrame(rows)\n\n    summary = (\n        raw.groupby("n_matches_per_sample", as_index=False)\n        .agg(\n            players=("players", "median"),\n            median_spearman=("spearman", "median"),\n            mean_spearman=("spearman", "mean"),\n            q025_spearman=("spearman", lambda x: np.nanquantile(x, 0.025)),\n            q975_spearman=("spearman", lambda x: np.nanquantile(x, 0.975)),\n            repeats=("rep", "size"),\n        )\n    )\n\n    summary["meets_target"] = summary["median_spearman"] >= STABILITY_TARGET\n\n    return fixed, summary, raw\n\n\ndef first_sustained_crossing(summary):\n    q = summary.sort_values("n_matches_per_sample").reset_index(drop=True)\n    flags = q["median_spearman"].ge(STABILITY_TARGET).to_numpy()\n\n    if len(flags) < SUSTAINED_POINTS:\n        return None\n\n    for i in range(0, len(flags) - SUSTAINED_POINTS + 1):\n        if bool(np.all(flags[i:i + SUSTAINED_POINTS])):\n            return int(q.loc[i, "n_matches_per_sample"])\n\n    return None\n\n\n# ================================================================\n# STEP 2 — Full-data match bootstrap\n# ================================================================\n\ndef bootstrap_full_cohort(pm, cohort, B=FULL_BOOTSTRAP_B, seed=SEED + 100):\n    player_ids = cohort["player_id"].tolist()\n    K = len(player_ids)\n\n    grouped = {\n        pid: q.reset_index(drop=True)\n        for pid, q in pm[pm["player_id"].isin(player_ids)].groupby("player_id")\n    }\n\n    V = np.empty((B, K), dtype=float)\n\n    rng = np.random.default_rng(seed)\n\n    for j, pid in enumerate(\n        tqdm(player_ids, desc="Full-cohort match bootstrap", unit="player")\n    ):\n        q = grouped[pid]\n        va = q["vaep"].to_numpy(dtype=float)\n        mi = q["minutes_played"].to_numpy(dtype=float)\n        m = len(q)\n\n        idx = rng.integers(0, m, size=(B, m))\n        total_v = va[idx].sum(axis=1)\n        total_m = mi[idx].sum(axis=1)\n\n        V[:, j] = (\n            90.0 * total_v / np.where(total_m > 0, total_m, np.nan)\n        )\n\n    if not np.isfinite(V).all():\n        raise RuntimeError("Non-finite bootstrap VAEP/90 values detected.")\n\n    # Rank every bootstrap leaderboard across all eligible players.\n    rank_draws = (\n        pd.DataFrame(V)\n        .rank(axis=1, ascending=False, method="average")\n        .to_numpy(dtype=float)\n    )\n\n    return player_ids, V, rank_draws\n\n\ndef pairwise_probability_matrix(V):\n    V = np.asarray(V, dtype=float)\n    K = V.shape[1]\n    P = np.full((K, K), 0.5, dtype=float)\n\n    for i in tqdm(range(K), desc="Pairwise probabilities", unit="player"):\n        for j in range(i + 1, K):\n            d = V[:, i] - V[:, j]\n            p = float(\n                np.mean(\n                    np.where(\n                        d > 0,\n                        1.0,\n                        np.where(d == 0, 0.5, 0.0),\n                    )\n                )\n            )\n            P[i, j] = p\n            P[j, i] = 1.0 - p\n\n    return P\n\n\n# ================================================================\n# STEP 3 — PCS / Bradley-Terry consensus\n# ================================================================\n\ndef fit_pairwise_consensus(P, ridge=BT_RIDGE):\n    P = np.asarray(P, dtype=float)\n    K = P.shape[0]\n\n    if K == 1:\n        return np.array([0.0]), {\n            "success": True,\n            "pairwise_mae": 0.0,\n            "pairwise_rmse": 0.0,\n            "objective": 0.0,\n        }\n\n    ii, jj = np.triu_indices(K, k=1)\n    target = P[ii, jj]\n    valid = np.isfinite(target)\n    ii, jj, target = ii[valid], jj[valid], target[valid]\n\n    def loss_grad(s):\n        z = s[ii] - s[jj]\n        losses = np.logaddexp(0.0, z) - target * z\n        loss = float(np.mean(losses) + ridge * np.mean(s ** 2))\n\n        err = (expit(z) - target) / len(target)\n        grad = np.zeros_like(s)\n        np.add.at(grad, ii, err)\n        np.add.at(grad, jj, -err)\n        grad += (2.0 * ridge / K) * s\n\n        return loss, grad\n\n    result = minimize(\n        fun=lambda s: loss_grad(s)[0],\n        x0=np.zeros(K, dtype=float),\n        jac=lambda s: loss_grad(s)[1],\n        method="L-BFGS-B",\n        options={"maxiter": 1000, "ftol": 1e-12},\n    )\n\n    score = result.x - np.mean(result.x)\n    pred = expit(score[ii] - score[jj])\n\n    return score, {\n        "success": bool(result.success),\n        "pairwise_mae": float(np.mean(np.abs(pred - target))),\n        "pairwise_rmse": float(np.sqrt(np.mean((pred - target) ** 2))),\n        "objective": float(result.fun),\n    }\n\n\n# ================================================================\n# STEP 4 — Full static VAEP vs interval vs RAR table\n# ================================================================\n\ndef build_static_ranking_table(pm, cohort):\n    player_ids, V, rank_draws = bootstrap_full_cohort(\n        pm,\n        cohort,\n        B=FULL_BOOTSTRAP_B,\n        seed=SEED + 100,\n    )\n\n    P = pairwise_probability_matrix(V)\n    pcs, fit_info = fit_pairwise_consensus(P)\n\n    rar = ranks_desc(pcs)\n    K = len(player_ids)\n\n    q025_rank = np.quantile(rank_draws, 0.025, axis=0)\n    q50_rank = np.quantile(rank_draws, 0.50, axis=0)\n    q975_rank = np.quantile(rank_draws, 0.975, axis=0)\n\n    rows = []\n\n    cohort_ix = cohort.set_index("player_id")\n\n    for i, pid in enumerate(player_ids):\n        p_i = np.delete(P[i, :], i)\n\n        strongly_above = int(\n            np.sum(p_i <= (1.0 - PAIRWISE_SUPPORT))\n        )\n        strongly_below = int(\n            np.sum(p_i >= PAIRWISE_SUPPORT)\n        )\n\n        supported_low = 1 + strongly_above\n        supported_high = K - strongly_below\n\n        resolved = (\n            (p_i >= PAIRWISE_SUPPORT)\n            | (p_i <= (1.0 - PAIRWISE_SUPPORT))\n        )\n\n        c = cohort_ix.loc[pid]\n\n        rows.append({\n            "player_id": pid,\n            "player_name": c["player_name"],\n            "is_goalkeeper": bool(c["is_goalkeeper"]),\n            "matches": int(c["total_matches"]),\n            "minutes": float(c["total_minutes"]),\n            "actions": int(c["total_actions"]),\n            "VAEP90": float(c["vaep90"]),\n            "VAEP_rank": int(c["raw_VAEP_rank"]),\n            "bootstrap_median_rank": float(q50_rank[i]),\n            "bootstrap_rank_low_2.5pct": float(q025_rank[i]),\n            "bootstrap_rank_high_97.5pct": float(q975_rank[i]),\n            "bootstrap_rank_interval_width": float(q975_rank[i] - q025_rank[i]),\n            "supported_rank_low": int(supported_low),\n            "supported_rank_high": int(supported_high),\n            "supported_rank_width": int(supported_high - supported_low),\n            "pairwise_resolution_rate": float(np.mean(resolved)),\n            "PCS": float(pcs[i]),\n            "RAR": int(rar[i]),\n            "RAR_minus_VAEP_rank": int(rar[i] - c["raw_VAEP_rank"]),\n            "absolute_rank_shift": int(abs(rar[i] - c["raw_VAEP_rank"])),\n        })\n\n    table = pd.DataFrame(rows)\n\n    # Raw ordering and RAR ordering are both preserved in the same table.\n    table = table.sort_values(\n        ["VAEP_rank", "RAR"]\n    ).reset_index(drop=True)\n\n    # Overall diagnostics\n    median_interval_width = float(\n        table["bootstrap_rank_interval_width"].median()\n    )\n    median_supported_width = float(\n        table["supported_rank_width"].median()\n    )\n    changed_fraction = float(\n        np.mean(table["VAEP_rank"] != table["RAR"])\n    )\n    changed_count = int(\n        np.sum(table["VAEP_rank"] != table["RAR"])\n    )\n\n    pair_i, pair_j = np.triu_indices(K, k=1)\n    pair_probs = P[pair_i, pair_j]\n\n    supported_pair_fraction = float(\n        np.mean(\n            (pair_probs >= PAIRWISE_SUPPORT)\n            | (pair_probs <= (1.0 - PAIRWISE_SUPPORT))\n        )\n    )\n\n    diagnostics = pd.DataFrame([{\n        "dataset": DATASET_LABEL,\n        "eligible_players": K,\n        "bootstrap_draws": FULL_BOOTSTRAP_B,\n        "median_95pct_rank_interval_width": median_interval_width,\n        "median_supported_rank_band_width": median_supported_width,\n        "supported_pair_fraction_90pct": supported_pair_fraction,\n        "players_reordered_by_RAR": changed_count,\n        "fraction_players_reordered_by_RAR": changed_fraction,\n        "median_absolute_RAR_shift": float(table["absolute_rank_shift"].median()),\n        "max_absolute_RAR_shift": int(table["absolute_rank_shift"].max()),\n        "BT_fit_success": fit_info["success"],\n        "BT_pairwise_mae": fit_info["pairwise_mae"],\n        "BT_pairwise_rmse": fit_info["pairwise_rmse"],\n    }])\n\n    # Pairwise output file.\n    pair_rows = []\n    names = cohort_ix["player_name"].to_dict()\n\n    for i in range(K):\n        for j in range(i + 1, K):\n            p = float(P[i, j])\n            if p >= PAIRWISE_SUPPORT:\n                status = "i_above_j_supported"\n            elif p <= (1.0 - PAIRWISE_SUPPORT):\n                status = "j_above_i_supported"\n            else:\n                status = "unresolved"\n\n            pair_rows.append({\n                "player_i_id": player_ids[i],\n                "player_i_name": names[player_ids[i]],\n                "player_j_id": player_ids[j],\n                "player_j_name": names[player_ids[j]],\n                "P_i_above_j": p,\n                "support_status": status,\n                "BT_pred_P_i_above_j": float(expit(pcs[i] - pcs[j])),\n            })\n\n    pairs = pd.DataFrame(pair_rows)\n\n    return table, diagnostics, pairs\n\n\n# ================================================================\n# STEP 5 — Independent raw VAEP vs RAR reproducibility stress test\n# ================================================================\n\ndef bootstrap_fixed_sample(q, B, rng):\n    q = q.reset_index(drop=True)\n    va = q["vaep"].to_numpy(dtype=float)\n    mi = q["minutes_played"].to_numpy(dtype=float)\n    n = len(q)\n\n    idx = rng.integers(0, n, size=(B, n))\n    total_v = va[idx].sum(axis=1)\n    total_m = mi[idx].sum(axis=1)\n\n    return 90.0 * total_v / np.where(total_m > 0, total_m, np.nan)\n\n\ndef rar_from_draws(draws):\n    P = pairwise_probability_matrix_fast(draws)\n    pcs, fit_info = fit_pairwise_consensus(P)\n    rar = ranks_desc(pcs)\n    return rar, fit_info\n\n\ndef pairwise_probability_matrix_fast(V):\n    """\n    Same calculation as the full matrix, without progress bars.\n    Used inside repeated validation loops.\n    """\n    V = np.asarray(V, dtype=float)\n    K = V.shape[1]\n    P = np.full((K, K), 0.5, dtype=float)\n\n    for i in range(K):\n        xi = V[:, i]\n        for j in range(i + 1, K):\n            d = xi - V[:, j]\n            p = float(\n                np.mean(\n                    np.where(\n                        d > 0,\n                        1.0,\n                        np.where(d == 0, 0.5, 0.0),\n                    )\n                )\n            )\n            P[i, j] = p\n            P[j, i] = 1.0 - p\n\n    return P\n\n\ndef run_independent_reproducibility(pm, cohort):\n    validation_cohort = cohort[\n        cohort["total_matches"] >= VALIDATION_MIN_MATCHES\n    ].copy()\n\n    player_ids = validation_cohort["player_id"].tolist()\n\n    print("\\n" + "=" * 96)\n    print("INDEPENDENT REPRODUCIBILITY STRESS TEST")\n    print("=" * 96)\n    print("Fixed validation cohort size:", len(player_ids))\n    print("Minimum observed matches in fixed cohort:", int(validation_cohort["total_matches"].min()))\n    print("n values:", VALIDATION_N_VALUES)\n    print("repetitions per n:", VALIDATION_REPS)\n    print("inner bootstrap draws per sample:", INNER_BOOTSTRAP_B)\n\n    if len(player_ids) < 50:\n        print(\n            "WARNING: validation cohort has fewer than 50 players. "\n            "The test remains valid but is less of a large-leaderboard stress test."\n        )\n\n    min_m = int(validation_cohort["total_matches"].min())\n    invalid_n = [n for n in VALIDATION_N_VALUES if 2 * n > min_m]\n    if invalid_n:\n        raise RuntimeError(\n            f"Validation n values {invalid_n} are impossible because the fixed cohort "\n            f"has a minimum of only {min_m} matches."\n        )\n\n    grouped = {\n        pid: (\n            q.sort_values(["game_date", "game_id"])\n            .reset_index(drop=True)\n        )\n        for pid, q in pm[pm["player_id"].isin(player_ids)].groupby("player_id")\n    }\n\n    rng_master = np.random.default_rng(SEED + 7000)\n    raw_rows = []\n\n    for n in VALIDATION_N_VALUES:\n        print(f"\\nTesting n={n} matches per independent sample...")\n\n        for rep in tqdm(\n            range(VALIDATION_REPS),\n            desc=f"n={n}",\n            unit="rep",\n        ):\n            K = len(player_ids)\n            raw_A = np.empty(K, dtype=float)\n            raw_B = np.empty(K, dtype=float)\n            draws_A = np.empty((INNER_BOOTSTRAP_B, K), dtype=float)\n            draws_B = np.empty((INNER_BOOTSTRAP_B, K), dtype=float)\n\n            for j, pid in enumerate(player_ids):\n                q = grouped[pid]\n\n                chosen = rng_master.choice(\n                    len(q),\n                    size=2 * n,\n                    replace=False,\n                )\n\n                qA = q.iloc[chosen[:n]].copy()\n                qB = q.iloc[chosen[n:]].copy()\n\n                raw_A[j] = vaep90(qA)\n                raw_B[j] = vaep90(qB)\n\n                rng_A = np.random.default_rng(\n                    SEED + 10_000_000 + n * 100_000 + rep * 1000 + j * 2\n                )\n                rng_B = np.random.default_rng(\n                    SEED + 20_000_000 + n * 100_000 + rep * 1000 + j * 2 + 1\n                )\n\n                draws_A[:, j] = bootstrap_fixed_sample(\n                    qA, INNER_BOOTSTRAP_B, rng_A\n                )\n                draws_B[:, j] = bootstrap_fixed_sample(\n                    qB, INNER_BOOTSTRAP_B, rng_B\n                )\n\n            raw_rank_A = ranks_desc(raw_A)\n            raw_rank_B = ranks_desc(raw_B)\n\n            rar_A, fit_A = rar_from_draws(draws_A)\n            rar_B, fit_B = rar_from_draws(draws_B)\n\n            rho_raw = float(\n                spearmanr(raw_rank_A, raw_rank_B).statistic\n            )\n            rho_rar = float(\n                spearmanr(rar_A, rar_B).statistic\n            )\n\n            raw_rows.append({\n                "n_matches_per_sample": n,\n                "rep": rep,\n                "players": K,\n                "raw_VAEP_spearman": rho_raw,\n                "RAR_spearman": rho_rar,\n                "RAR_minus_raw": rho_rar - rho_raw,\n                "RAR_better": int(rho_rar > rho_raw),\n                "RAR_equal": int(np.isclose(rho_rar, rho_raw)),\n                "RAR_worse": int(rho_rar < rho_raw),\n                "BT_A_success": bool(fit_A["success"]),\n                "BT_B_success": bool(fit_B["success"]),\n            })\n\n    raw = pd.DataFrame(raw_rows)\n\n    summary_rows = []\n\n    for n, q in raw.groupby("n_matches_per_sample"):\n        delta = q["RAR_minus_raw"].to_numpy(dtype=float)\n        nonzero = delta[~np.isclose(delta, 0.0)]\n\n        if len(nonzero) == 0:\n            p_value = 1.0\n        else:\n            try:\n                p_value = float(\n                    wilcoxon(\n                        nonzero,\n                        alternative="two-sided",\n                        zero_method="wilcox",\n                    ).pvalue\n                )\n            except Exception:\n                p_value = np.nan\n\n        summary_rows.append({\n            "n_matches_per_sample": int(n),\n            "players": int(q["players"].iloc[0]),\n            "reps": len(q),\n            "median_raw_VAEP_spearman": float(q["raw_VAEP_spearman"].median()),\n            "median_RAR_spearman": float(q["RAR_spearman"].median()),\n            "median_RAR_minus_raw": float(np.median(delta)),\n            "mean_RAR_minus_raw": float(np.mean(delta)),\n            "q025_RAR_minus_raw": float(np.quantile(delta, 0.025)),\n            "q975_RAR_minus_raw": float(np.quantile(delta, 0.975)),\n            "fraction_RAR_better": float(np.mean(delta > 0)),\n            "fraction_equal": float(np.mean(np.isclose(delta, 0.0))),\n            "fraction_RAR_worse": float(np.mean(delta < 0)),\n            "paired_wilcoxon_p": p_value,\n        })\n\n    summary = pd.DataFrame(summary_rows)\n\n    overall = pd.DataFrame([{\n        "dataset": DATASET_LABEL,\n        "fixed_validation_players": len(player_ids),\n        "validation_minimum_matches_required": VALIDATION_MIN_MATCHES,\n        "tested_n_values": ",".join(str(x) for x in VALIDATION_N_VALUES),\n        "reps_per_n": VALIDATION_REPS,\n        "inner_bootstrap_draws": INNER_BOOTSTRAP_B,\n        "pooled_median_RAR_minus_raw": float(raw["RAR_minus_raw"].median()),\n        "pooled_mean_RAR_minus_raw": float(raw["RAR_minus_raw"].mean()),\n        "pooled_fraction_RAR_better": float(np.mean(raw["RAR_minus_raw"] > 0)),\n        "pooled_fraction_equal": float(np.mean(np.isclose(raw["RAR_minus_raw"], 0.0))),\n        "pooled_fraction_RAR_worse": float(np.mean(raw["RAR_minus_raw"] < 0)),\n        "n_points_RAR_higher_median": int(\n            np.sum(summary["median_RAR_spearman"] > summary["median_raw_VAEP_spearman"])\n        ),\n        "n_points_equal_median": int(\n            np.sum(\n                np.isclose(\n                    summary["median_RAR_spearman"],\n                    summary["median_raw_VAEP_spearman"],\n                )\n            )\n        ),\n        "n_points_RAR_lower_median": int(\n            np.sum(summary["median_RAR_spearman"] < summary["median_raw_VAEP_spearman"])\n        ),\n    }])\n\n    return validation_cohort, raw, summary, overall\n\n\n# ================================================================\n# RUN COMPLETE ANALYSIS\n# ================================================================\n\npm, base_cohort, model_metrics, build_meta = build_wsl_player_match()\n\nbase_cohort.to_csv(\n    OUT_ROOT / "00_base_eligible_league_cohort.csv",\n    index=False,\n)\n\n# ------------------------------------------------------------\n# Pre-RAR stability screen\n# ------------------------------------------------------------\nanalysis_cohort, stability_summary, stability_raw = longitudinal_raw_stability(\n    pm,\n    base_cohort,\n)\n\nstability_crossing = first_sustained_crossing(stability_summary)\nstability_demonstrated = stability_crossing is not None\n\nanalysis_cohort.to_csv(\n    OUT_ROOT / "01_longitudinal_fixed_cohort_45plus_matches.csv",\n    index=False,\n)\nstability_summary.to_csv(\n    OUT_ROOT / "02_raw_VAEP_stability_by_n.csv",\n    index=False,\n)\nstability_raw.to_csv(\n    OUT_ROOT / "03_raw_VAEP_stability_replicates.csv",\n    index=False,\n)\n\nprint("\\nRaw stability summary:")\nprint(stability_summary.round(4).to_string(index=False))\n\nif stability_demonstrated:\n    print(\n        f"\\nSustained median Spearman >= {STABILITY_TARGET:.2f} begins at "\n        f"n={stability_crossing} matches per independent sample."\n    )\nelse:\n    print(\n        f"\\nNo sustained median Spearman >= {STABILITY_TARGET:.2f} crossing was "\n        "observed in the tested n range. Do NOT label this cohort stable."\n    )\n\n# ------------------------------------------------------------\n# Full ReliableVAEP / PCS / RAR on the SAME fixed cohort\n# ------------------------------------------------------------\nstatic_table, static_diagnostics, pairwise_table = build_static_ranking_table(\n    pm,\n    analysis_cohort,\n)\n\nstatic_table.to_csv(\n    OUT_ROOT / "04_FULL_VAEP_vs_RAR_with_intervals.csv",\n    index=False,\n)\n\nstatic_table.sort_values(\n    ["RAR", "VAEP_rank"]\n).to_csv(\n    OUT_ROOT / "05_FULL_RAR_ordered_leaderboard.csv",\n    index=False,\n)\n\nstatic_table.sort_values(\n    ["VAEP_rank", "RAR"]\n).to_csv(\n    OUT_ROOT / "06_FULL_VAEP_ordered_leaderboard.csv",\n    index=False,\n)\n\nstatic_table.sort_values(\n    ["absolute_rank_shift", "VAEP_rank"],\n    ascending=[False, True],\n).to_csv(\n    OUT_ROOT / "07_players_with_largest_RAR_changes.csv",\n    index=False,\n)\n\npairwise_table.to_csv(\n    OUT_ROOT / "08_pairwise_probabilities.csv",\n    index=False,\n)\n\nstatic_diagnostics.to_csv(\n    OUT_ROOT / "09_static_diagnostics.csv",\n    index=False,\n)\n\nprint("\\n" + "=" * 96)\nprint("STATIC WSL LONGITUDINAL LEAGUE RESULT")\nprint("=" * 96)\nprint(static_diagnostics.round(4).to_string(index=False))\n\nprint("\\nTop 30 by original VAEP rank:")\nprint(\n    static_table.sort_values("VAEP_rank")[\n        [\n            "VAEP_rank",\n            "player_name",\n            "VAEP90",\n            "bootstrap_rank_low_2.5pct",\n            "bootstrap_rank_high_97.5pct",\n            "supported_rank_low",\n            "supported_rank_high",\n            "PCS",\n            "RAR",\n            "RAR_minus_VAEP_rank",\n        ]\n    ]\n    .head(30)\n    .round(4)\n    .to_string(index=False)\n)\n\nprint("\\nTop 30 by RAR:")\nprint(\n    static_table.sort_values(["RAR", "VAEP_rank"])[\n        [\n            "RAR",\n            "player_name",\n            "PCS",\n            "VAEP_rank",\n            "VAEP90",\n            "bootstrap_rank_low_2.5pct",\n            "bootstrap_rank_high_97.5pct",\n            "supported_rank_low",\n            "supported_rank_high",\n            "RAR_minus_VAEP_rank",\n        ]\n    ]\n    .head(30)\n    .round(4)\n    .to_string(index=False)\n)\n\n# ------------------------------------------------------------\n# Independent RAR validation on the SAME fixed cohort\n# ------------------------------------------------------------\nvalidation_cohort, validation_raw, validation_summary, validation_overall = (\n    run_independent_reproducibility(pm, analysis_cohort)\n)\n\nvalidation_cohort.to_csv(\n    OUT_ROOT / "10_fixed_RAR_validation_cohort.csv",\n    index=False,\n)\nvalidation_raw.to_csv(\n    OUT_ROOT / "11_RAR_vs_raw_validation_replicates.csv",\n    index=False,\n)\nvalidation_summary.to_csv(\n    OUT_ROOT / "12_RAR_vs_raw_validation_by_n.csv",\n    index=False,\n)\nvalidation_overall.to_csv(\n    OUT_ROOT / "13_RAR_vs_raw_validation_overall.csv",\n    index=False,\n)\n\nprint("\\n" + "=" * 96)\nprint("RAW VAEP VS RAR REPRODUCIBILITY")\nprint("=" * 96)\nprint(validation_summary.round(4).to_string(index=False))\n\nprint("\\nOverall validation:")\nprint(validation_overall.round(4).to_string(index=False))\n\n\n# ================================================================\n# Figures\n# ================================================================\n\n# Figure 1: raw stability curve.\nfig, ax = plt.subplots(figsize=(8.5, 5.5))\nax.plot(\n    stability_summary["n_matches_per_sample"],\n    stability_summary["median_spearman"],\n    marker="o",\n    label="Median raw VAEP Spearman",\n)\nax.fill_between(\n    stability_summary["n_matches_per_sample"].to_numpy(dtype=float),\n    stability_summary["q025_spearman"].to_numpy(dtype=float),\n    stability_summary["q975_spearman"].to_numpy(dtype=float),\n    alpha=0.18,\n    label="2.5-97.5% replicate range",\n)\nax.axhline(\n    STABILITY_TARGET,\n    linestyle="--",\n    label=f"{STABILITY_TARGET:.2f} stability target",\n)\nax.set_xlabel("Matches per independent sample")\nax.set_ylabel("Raw VAEP rank Spearman")\nax.set_title("WSL 2018/19-2020/21 longitudinal league stability")\nax.set_ylim(-1, 1)\nax.legend()\nplt.tight_layout()\nfig.savefig(\n    OUT_ROOT / "20_raw_VAEP_stability_curve.png",\n    dpi=180,\n)\nplt.close(fig)\n\n# Figure 2: raw VAEP rank vs RAR.\nq = static_table.sort_values("VAEP_rank").copy()\nfig, ax = plt.subplots(figsize=(8, 6))\nax.scatter(q["VAEP_rank"], q["RAR"])\nmax_rank = max(q["VAEP_rank"].max(), q["RAR"].max())\nax.plot([1, max_rank], [1, max_rank], linestyle="--")\nax.set_xlabel("Original VAEP rank")\nax.set_ylabel("RAR")\nax.set_title("WSL longitudinal cohort: VAEP rank vs RAR")\nax.invert_xaxis()\nax.invert_yaxis()\nplt.tight_layout()\nfig.savefig(\n    OUT_ROOT / "21_VAEP_rank_vs_RAR.png",\n    dpi=180,\n)\nplt.close(fig)\n\n# Figure 3: interval width by original rank.\nfig, ax = plt.subplots(figsize=(8, 5.5))\nax.scatter(\n    static_table["VAEP_rank"],\n    static_table["bootstrap_rank_interval_width"],\n)\nax.set_xlabel("Original VAEP rank")\nax.set_ylabel("95% bootstrap rank-interval width")\nax.set_title("WSL longitudinal cohort ranking uncertainty")\nplt.tight_layout()\nfig.savefig(\n    OUT_ROOT / "22_bootstrap_interval_width_by_rank.png",\n    dpi=180,\n)\nplt.close(fig)\n\n# Figure 4: independent raw vs RAR reproducibility.\nfig, ax = plt.subplots(figsize=(8, 5.5))\nax.plot(\n    validation_summary["n_matches_per_sample"],\n    validation_summary["median_raw_VAEP_spearman"],\n    marker="o",\n    label="Raw VAEP rank",\n)\nax.plot(\n    validation_summary["n_matches_per_sample"],\n    validation_summary["median_RAR_spearman"],\n    marker="o",\n    label="RAR",\n)\nax.axhline(STABILITY_TARGET, linestyle=":", label=f"{STABILITY_TARGET:.2f} reference")\nax.set_xlabel("Matches per independent sample")\nax.set_ylabel("Median Spearman reproducibility")\nax.set_title("WSL longitudinal league: raw VAEP vs RAR")\nax.set_ylim(-1, 1)\nax.legend()\nplt.tight_layout()\nfig.savefig(\n    OUT_ROOT / "23_raw_VAEP_vs_RAR_reproducibility.png",\n    dpi=180,\n)\nplt.close(fig)\n\n# Figure 5: RAR gain.\nfig, ax = plt.subplots(figsize=(8, 5.5))\nax.plot(\n    validation_summary["n_matches_per_sample"],\n    validation_summary["median_RAR_minus_raw"],\n    marker="o",\n)\nax.axhline(0.0, linestyle="--")\nax.set_xlabel("Matches per independent sample")\nax.set_ylabel("Median Spearman gain (RAR - raw VAEP)")\nax.set_title("WSL longitudinal cohort: RAR reproducibility gain")\nplt.tight_layout()\nfig.savefig(\n    OUT_ROOT / "24_RAR_reproducibility_gain.png",\n    dpi=180,\n)\nplt.close(fig)\n\n\n# ================================================================\n# Paper-facing summary\n# ================================================================\n\ndiag = static_diagnostics.iloc[0]\nov = validation_overall.iloc[0]\n\ncrossing_text = (\n    str(stability_crossing)\n    if stability_crossing is not None\n    else "none in tested range"\n)\n\nsummary_lines = [\n    "ReliableVAEP WSL 2018/19-2020/21 longitudinal league experiment",\n    "=" * 76,\n    "",\n    f"StatsBomb matches analysed: {build_meta[\'statsbomb_matches\']}",\n    f"Official played matches across selected seasons: {build_meta[\'official_played_matches\']}",\n    f"Observed StatsBomb source coverage: {100*build_meta[\'source_coverage_fraction\']:.1f}%",\n    f"Base eligible players: {len(base_cohort)}",\n    f"Longitudinal >=45-match ranking cohort: {len(analysis_cohort)} players",\n    f"Raw stability target: median Spearman >= {STABILITY_TARGET:.2f} "\n    f"for {SUSTAINED_POINTS} consecutive n values",\n    f"First sustained crossing: {crossing_text}",\n    f"Stable under declared operational rule: {stability_demonstrated}",\n    "",\n    f"Median 95% bootstrap rank-interval width: "\n    f"{diag[\'median_95pct_rank_interval_width\']:.2f}",\n    f"Median supported rank-band width: "\n    f"{diag[\'median_supported_rank_band_width\']:.2f}",\n    f"Supported pair fraction at 90%: "\n    f"{100*diag[\'supported_pair_fraction_90pct\']:.1f}%",\n    f"Players reordered by RAR: {int(diag[\'players_reordered_by_RAR\'])} "\n    f"({100*diag[\'fraction_players_reordered_by_RAR\']:.1f}%)",\n    f"Median absolute RAR shift: {diag[\'median_absolute_RAR_shift\']:.2f} ranks",\n    f"Maximum absolute RAR shift: {int(diag[\'max_absolute_RAR_shift\'])} ranks",\n    "",\n    "Independent RAR reproducibility validation:",\n    f"Fixed validation players: {int(ov[\'fixed_validation_players\'])}",\n    f"Pooled median Spearman gain (RAR - raw): "\n    f"{ov[\'pooled_median_RAR_minus_raw\']:.4f}",\n    f"Pooled mean Spearman gain (RAR - raw): "\n    f"{ov[\'pooled_mean_RAR_minus_raw\']:.4f}",\n    f"Replicates RAR better/equal/worse: "\n    f"{ov[\'pooled_fraction_RAR_better\']:.3f} / "\n    f"{ov[\'pooled_fraction_equal\']:.3f} / "\n    f"{ov[\'pooled_fraction_RAR_worse\']:.3f}",\n    "",\n    "Interpretation:",\n    "Call this the stable-league arm only if the pre-RAR raw-VAEP curve reaches "\n    "the declared sustained 0.80 criterion. If it does not, report it as another "\n    "league-level instability result rather than changing the threshold after seeing "\n    "the data. RAR should be described as beneficial only if independent A/B "\n    "reproducibility shows a consistent positive gain.",\n]\n\nsummary_text = "\\n".join(summary_lines)\n(OUT_ROOT / "25_paper_facing_summary.txt").write_text(\n    summary_text,\n    encoding="utf-8",\n)\n\nprint("\\n" + summary_text)\n\nzip_path = shutil.make_archive(\n    str(ROOT / "ReliableVAEP_WSL_1819_2021_LONGITUDINAL_RESULTS"),\n    "zip",\n    root_dir=OUT_ROOT,\n)\n\nprint("\\n" + "=" * 96)\nprint("ANALYSIS COMPLETE")\nprint("=" * 96)\nprint("Results folder:", OUT_ROOT)\nprint("Results ZIP:", zip_path)\n'

SCRIPT_PATH.write_text(ANALYSIS_SCRIPT, encoding="utf-8")
compile(ANALYSIS_SCRIPT, str(SCRIPT_PATH), "exec")

print("Analysis script written to:", SCRIPT_PATH)
print("Syntax check: OK")
print("Script size:", len(ANALYSIS_SCRIPT), "bytes")


Analysis script written to: /kaggle/working/reliablevaep_wsl_longitudinal_analysis.py
Syntax check: OK
Script size: 57194 bytes


In [4]:
# ================================================================
# CELL 4 — RUN THE COMPLETE ANALYSIS
# ================================================================
import os
import subprocess
from pathlib import Path

ENV_PY = Path("/kaggle/working/rv_vaep_env/bin/python")
SCRIPT_PATH = Path("/kaggle/working/reliablevaep_wsl_longitudinal_analysis.py")

if not ENV_PY.exists():
    raise RuntimeError(
        "Run Cell 2 first. The isolated Kaggle VAEP environment does not exist."
    )

if not SCRIPT_PATH.exists():
    raise RuntimeError(
        "Run Cell 3 first. The analysis script does not exist."
    )

child_env = os.environ.copy()
child_env["MPLBACKEND"] = "Agg"
child_env.pop("MATPLOTLIBRC", None)
child_env.pop("PYTHONPATH", None)
child_env["PYTHONNOUSERSITE"] = "1"

print("Running WSL longitudinal ReliableVAEP / RAR analysis...")
print("Analysis Python:", ENV_PY)
print()

process = subprocess.Popen(
    [str(ENV_PY), "-u", str(SCRIPT_PATH)],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
    env=child_env,
)

for line in process.stdout:
    print(line, end="")

return_code = process.wait()

if return_code != 0:
    raise RuntimeError(
        f"Analysis exited with code {return_code}. "
        "The exact traceback is shown immediately above."
    )

print("\nWSL longitudinal analysis finished successfully.")


Running WSL longitudinal ReliableVAEP / RAR analysis...
Analysis Python: /kaggle/working/rv_vaep_env/bin/python

ReliableVAEP WSL 2018/19-2020/21 longitudinal league stability + PCS/RAR
Competition ID: 37
Seasons: 2018/2019, 2019/2020, 2020/2021
Official played matches for coverage reference: 328
Minimum accepted StatsBomb source coverage: 95.0%
Base eligibility: >=5 matches, >=300 minutes, >=150 actions
Longitudinal ranking cohort: >= 45 appearances
Raw-stability target: median Spearman >= 0.80 for 3 consecutive n values
Full bootstrap draws: 3500
Pairwise support threshold: 0.9
RAR validation n values: [5, 10, 15, 20]
RAR validation repetitions per n: 200

WSL 2018/2019: StatsBomb matches found = 107 (official played = 110; coverage = 97.3%)
WSL 2019/2020: StatsBomb matches found = 87 (official played = 87; coverage = 100.0%)
WSL 2020/2021: StatsBomb matches found = 131 (official played = 131; coverage = 100.0%)
Coverage audit passed. StatsBomb sample = 325/328 official played matche

In [5]:
# ================================================================
# CELL 5 — REVIEW THE MAIN RESULTS
# ================================================================
import csv
import html
from pathlib import Path
from IPython.display import display, HTML

OUT = Path("/kaggle/working/rv_wsl_1819_2021_pcs_rar_outputs")

def read_csv(path):
    with open(path, newline="", encoding="utf-8") as f:
        return list(csv.DictReader(f))

def table(title, rows, columns=None, limit=80):
    if not rows:
        print(title, ": no rows")
        return
    rows = rows[:limit]
    if columns is None:
        columns = list(rows[0].keys())
    parts = [
        f"<h3>{html.escape(title)}</h3>",
        '<div style="overflow:auto;max-height:650px">',
        '<table style="border-collapse:collapse;width:100%;font-size:13px">',
        "<thead><tr>",
    ]
    for c in columns:
        parts.append(
            f'<th style="border:1px solid #ddd;padding:5px;position:sticky;top:0;'
            f'background:#f3f4f6">{html.escape(c)}</th>'
        )
    parts.append("</tr></thead><tbody>")
    for r in rows:
        parts.append("<tr>")
        for c in columns:
            parts.append(
                f'<td style="border:1px solid #ddd;padding:5px">'
                f'{html.escape(str(r.get(c, "")))}</td>'
            )
        parts.append("</tr>")
    parts.append("</tbody></table></div>")
    display(HTML("".join(parts)))

stability_path = OUT / "02_raw_VAEP_stability_by_n.csv"
ranking_path = OUT / "04_FULL_VAEP_vs_RAR_with_intervals.csv"
validation_path = OUT / "12_RAR_vs_raw_validation_by_n.csv"
summary_path = OUT / "25_paper_facing_summary.txt"

if stability_path.exists():
    table(
        "Pre-RAR raw VAEP stability",
        read_csv(stability_path),
        [
            "n_matches_per_sample", "players", "median_spearman",
            "q025_spearman", "q975_spearman", "meets_target"
        ],
    )

if validation_path.exists():
    table(
        "Independent raw VAEP vs RAR reproducibility",
        read_csv(validation_path),
        [
            "n_matches_per_sample", "players",
            "median_raw_VAEP_spearman", "median_RAR_spearman",
            "median_RAR_minus_raw", "fraction_RAR_better",
            "paired_wilcoxon_p"
        ],
    )

if ranking_path.exists():
    rows = read_csv(ranking_path)
    rows = sorted(rows, key=lambda r: float(r["VAEP_rank"]))
    table(
        "Longitudinal league cohort: VAEP → uncertainty → RAR",
        rows,
        [
            "VAEP_rank", "player_name", "VAEP90",
            "bootstrap_rank_low_2.5pct", "bootstrap_rank_high_97.5pct",
            "supported_rank_low", "supported_rank_high",
            "PCS", "RAR", "RAR_minus_VAEP_rank"
        ],
        limit=60,
    )

if summary_path.exists():
    print("\n" + "=" * 80)
    print(summary_path.read_text(encoding="utf-8"))


n_matches_per_sample,players,median_spearman,q025_spearman,q975_spearman,meets_target
5,46.0,0.3240826395312982,0.03611162503854456,0.569568300955905,False
6,46.0,0.3588652482269503,0.04338575393154487,0.5957600986740672,False
7,46.0,0.37971014492753624,0.12759790317607153,0.6191057662658034,False
8,46.0,0.4201665124884366,0.15531606537156953,0.6521831637372804,False
9,46.0,0.4495837187789084,0.2016867098365711,0.6509959913660192,False
10,46.0,0.48233117483811283,0.22171137835337645,0.6796238051187172,False
11,46.0,0.49071847055195805,0.2966358310206599,0.6640302189330867,False
12,46.0,0.5158186864014801,0.30333024976873263,0.6946623496762256,False
13,46.0,0.5317298797409806,0.28727104532839964,0.7056645081714462,False
14,46.0,0.5605920444033302,0.3831359851988899,0.7146438482886217,False


n_matches_per_sample,players,median_raw_VAEP_spearman,median_RAR_spearman,median_RAR_minus_raw,fraction_RAR_better,paired_wilcoxon_p
5,46,0.3240826395312982,0.35775516497070614,0.03398088189947578,0.835,3.594377428400189e-24
10,46,0.48245451742213996,0.4967622571692877,0.018501387604070246,0.77,1.5561268749330165e-17
15,46,0.5730496453900709,0.5860006167129201,0.011964230650632124,0.745,9.51083023449454e-16
20,46,0.6460684551341351,0.6508788159111933,0.008633980881899594,0.66,1.8230613111448597e-08


VAEP_rank,player_name,VAEP90,bootstrap_rank_low_2.5pct,bootstrap_rank_high_97.5pct,supported_rank_low,supported_rank_high,PCS,RAR,RAR_minus_VAEP_rank
1,Vivianne Miedema,0.6365928463095194,1.0,6.0,1,3,6.898440815237486,1,0
2,So-Yun Ji,0.5466172974977742,1.0,6.0,1,5,6.281857163927677,2,0
3,Caroline Weir,0.4512472198275431,1.0,13.0,1,11,4.780736758455101,3,0
4,Leah Williamson,0.39620945969086935,3.0,11.0,3,13,4.175720202607358,4,0
5,Danielle van de Donk,0.39366756224799704,2.0,20.0,2,16,3.6431069295358376,5,0
6,Georgia Stanway,0.3882574977673713,1.0,22.0,2,17,3.549056537190548,7,1
7,Katie McCabe,0.36871820141807043,3.0,15.0,3,15,3.5895963504359156,6,-1
8,Stephanie Houghton,0.3509013401891834,4.0,16.0,3,16,3.2604247802259625,8,0
9,Bethany Mead,0.33353130435834344,3.0,20.0,3,18,2.8464009485143524,10,1
10,Keira Walsh,0.3304629971401696,4.0,17.0,4,17,2.877859212488536,9,-1



ReliableVAEP WSL 2018/19-2020/21 longitudinal league experiment

StatsBomb matches analysed: 325
Official played matches across selected seasons: 328
Observed StatsBomb source coverage: 99.1%
Base eligible players: 331
Longitudinal >=45-match ranking cohort: 46 players
Raw stability target: median Spearman >= 0.80 for 3 consecutive n values
First sustained crossing: none in tested range
Stable under declared operational rule: False

Median 95% bootstrap rank-interval width: 18.26
Median supported rank-band width: 16.00
Supported pair fraction at 90%: 62.5%
Players reordered by RAR: 16 (34.8%)
Median absolute RAR shift: 0.00 ranks
Maximum absolute RAR shift: 3 ranks

Independent RAR reproducibility validation:
Fixed validation players: 46
Pooled median Spearman gain (RAR - raw): 0.0152
Pooled mean Spearman gain (RAR - raw): 0.0183
Replicates RAR better/equal/worse: 0.752 / 0.001 / 0.246

Interpretation:
Call this the stable-league arm only if the pre-RAR raw-VAEP curve reaches the decl

## Main outputs

After the Kaggle run, check these first:

- `coverage_audit.csv` in the cache dataset folder: exact source coverage by season.
- `02_raw_VAEP_stability_by_n.csv`: determines whether this can actually be called the stable-league arm.
- `04_FULL_VAEP_vs_RAR_with_intervals.csv`: VAEP, uncertainty, PCS, and RAR.
- `12_RAR_vs_raw_validation_by_n.csv`: independent raw-vs-RAR reproducibility.
- `25_paper_facing_summary.txt`: conservative summary using the **actual** StatsBomb sample size.
- `ReliableVAEP_WSL_1819_2021_LONGITUDINAL_RESULTS.zip`: complete outputs.

Do not call the dataset stable unless the raw-VAEP curve reaches the declared
sustained median Spearman ≥0.80 criterion.
